# Hybrid Quantum-Classical Reinforcement Learning (2/3)

___
___

## Introduction
___

This series of experiments is based on the 2025 paper by Nagy et al.: "[Hybrid Quantum-Classical Reinforcement Learning in Latent Observation Spaces](https://arxiv.org/abs/2410.18284)".

In this article, the authors apply hybrid quantum-classical Reinforcement Learning (RL) models to two simulated environments. They compare classical, qubit-based and photonic-based agents, all using Proximal Policy Optimization (PPO). They also use an AutoEncoder (AE) to compress the dimensionality of the observations and train that AE jointly with the PPO agents.

The notebook series aims to compare the resources cost of the different systems to reach the same performance. It is divided in three parts:

- [Part I: Classical vs Qubit agents on the Cart Pole environment](QRL_experiment_1.ipynb)
- **Part II: Classical vs Qubit agents on the Lunar Lander and Car Racing environments**
- [Part III: Photonic agents on the three environments](QRL_experiment_3.ipynb)

This second notebook will present:
1. The Lunar Lander environment
2. The Car Racing environment
3. Review of previous notebook's code
4. Results
5. Conclusions
6. Follow-up

In [ ]:
# !pip install ipynb torch torchvision swig gymnasium[box2d] matplotlib pennylane
import os
import random
import datetime
from itertools import count
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import cv2

from ipynb.fs.defs.QRL_experiment_1 import AutoEncoder, PPO, ActorNN, CriticNN, ActorQubitNN, count_parameters

# For reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

## 1 - The Lunar Lander environment
___

In this Box2D environment, the agent learns to land a spaceship in a designated area. 

<div align="center">
<img src="images/qrl_demo_lunarlander.gif" width="300"/>
</div>

At each step, the agent gets 8 observations: the x and y coordinates of the ship, its x and y linear velocities, its angle, angular velocity, and whether each leg of the ship is touching the ground. The agent can choose between 4 actions: do nothing, fire the main engine, fire the left engine or fire the right one.

A reward is granted at each step, depending on the ship's position compared to the landing pad, its velocity, orientation, engine firings and contact with the ground. An additional reward is given at the end depending on whether the ship landed correctly or not. The episode ends when the ship crashes, is getting out of display or stalls.

More information about this environment can be found here: https://gymnasium.farama.org/environments/box2d/lunar_lander/.

In [ ]:
lunarlander_env = gym.make("LunarLander-v3", render_mode="rgb_array")
lunarlander_env.reset(seed=seed)
lunarlander_env.action_space.seed(seed)
lunarlander_env.observation_space.seed(seed)

## 2 - The Car Racing environment
___

In this Box2D environment, the agent learns to drive a car on a circuit. 

<div align="center">
<img src="images/qrl_demo_carracing.gif" width="300"/>
</div>

At each step, the agent gets a 96x96 RGB image as observations. It can then choose between 5 actions: do nothing, steer right, steer left,  gas or brake.

The reward is -0.1 at every step and +1000/N for every track tile visited, where N is the total number of tiles visited in the track. For example, if you have finished in 732 frames, your reward is 1000 - 0.1*732 = 926.8 points. The episode finishes when all the tiles are visited. The car can also go outside the playfield - that is, far off the track, in which case it will receive -100 reward and die.

More information about this environment can be found here: https://gymnasium.farama.org/environments/box2d/car_racing/.

In [ ]:
carracing_env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
carracing_env.reset(seed=seed)
carracing_env.action_space.seed(seed)
carracing_env.observation_space.seed(seed)

## 3 - Reused code from the first notebook
___

The first notebook introduced AutoEncoders, as well as classical and qubit Proximal Policy Optimization (PPO) agents.

For the Car Racing env however, since the input is an image, we will use an AutoEncoder and Critic model with Convolutional layers.

Convolutions

In [ ]:
class CNN(nn.Module):
    def __init__(self, input_shape, hidden_dims, kernel_size=4, stride_size=2, padding=1):
        super().__init__()
        width, _, input_channels = input_shape
        self.conv1 = nn.Conv2d(input_channels, hidden_dims[0], kernel_size, stride_size, padding)
        conv1_size = int((width+2*padding-kernel_size)/stride_size+1)
        self.conv2 = nn.Conv2d(hidden_dims[0], hidden_dims[1], kernel_size, stride_size, padding)
        conv2_size = int((conv1_size+2*padding-kernel_size)/stride_size+1)
        self.conv3 = nn.Conv2d(hidden_dims[1], hidden_dims[2], kernel_size, stride_size, padding)
        conv3_size = int((conv2_size+2*padding-kernel_size)/stride_size+1)
        self.flatten = nn.Flatten()
        self.output_size = hidden_dims[2] * conv3_size**2

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return self.flatten(x)
    
class CriticCNN(nn.Module):
    def __init__(self, input_shape, intermediate_dim, output_dim):
        super().__init__()
        width, _, input_channels = input_shape
        self.cnn = CNN(input_shape, intermediate_dim)
        self.layer1 = nn.Linear(self.cnn.output_size, intermediate_dim[-1])
        self.layer2 = nn.Linear(intermediate_dim[-1], intermediate_dim[-1])
        self.layer3 = nn.Linear(intermediate_dim[-1], output_dim)

    def forward(self, x):
        x = self.cnn(x)
        x = self.layer1(x)
        x = F.relu(x)
        x = self.layer2(x)
        x = F.relu(x)
        return self.layer3(x)

class EncoderCNN(nn.Module):
    def __init__(self, input_shape, hidden_dims, output_dim):
        super().__init__()
        width, _, input_channels = input_shape
        self.cnn = CNN(input_shape, hidden_dims)
        self.fc1 = nn.Linear(self.cnn.output_size, hidden_dims[-1])
        self.fc2 = nn.Linear(hidden_dims[-1], hidden_dims[-1])
        self.fc3 = nn.Linear(hidden_dims[-1], output_dim)

    def forward(self, x):
        x = self.cnn(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        return self.fc3(x)

class DecoderCNN(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_shape, kernel_size=4, stride_size=2, padding=1):
        super().__init__()
        width, _, output_channels = output_shape
        self.conv1 = nn.ConvTranspose2d(hidden_dims[0], output_channels, kernel_size, stride_size, padding)
        conv1_size = int((width+2*padding-kernel_size)/stride_size+1)
        self.conv2 = nn.ConvTranspose2d(hidden_dims[1], hidden_dims[0], kernel_size, stride_size, padding)
        conv2_size = int((conv1_size+2*padding-kernel_size)/stride_size+1)
        self.conv3 = nn.ConvTranspose2d(hidden_dims[2], hidden_dims[1], kernel_size, stride_size, padding)
        conv3_size = int((conv2_size+2*padding-kernel_size)/stride_size+1)
        self.intermediate_shape = (-1, hidden_dims[2], conv3_size, conv3_size)
        self.fc1 = nn.Linear(hidden_dims[3], hidden_dims[2] * conv3_size**2)
        self.fc2 = nn.Linear(hidden_dims[3], hidden_dims[3])
        self.fc3 = nn.Linear(input_dim, hidden_dims[3])
        
    def forward(self, x):
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc2(x))
        x = self.fc1(x)
        x = x.view(self.intermediate_shape)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv2(x))
        x = F.sigmoid(self.conv1(x))
        return x.permute(0, 2, 3, 1)

class ConvolutionalAE(nn.Module):
    def __init__(self, input_shape, parameters):
        super().__init__()
        self.image_shape = input_shape
        self.encoder = EncoderCNN(self.image_shape, parameters["ae_hidden_dims"], parameters["ae_output_dim"])
        self.decoder = DecoderCNN(parameters["ae_output_dim"], parameters["ae_hidden_dims"], self.image_shape)
        self.hyperparameters = parameters

    def preprocess(self, stacked_states):
        stacked_states = np.stack(
            [cv2.resize(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), self.image_shape[:2]) for img in stacked_states],
            axis=-1
        )
        stacked_states = stacked_states / 255.0
        return stacked_states
    
    def forward(self, x):
        x = self.encoder(x)
        x = F.tanh(x)
        x = self.decoder(x)
        return x
    
    def pre_train(self, env, device):
        print("Pre-training AE")
        optimizer = optim.Adam(self.parameters())
        epochs = self.hyperparameters["ae_pretrain_epochs"]
        batch_size = self.hyperparameters["ae_pretrain_batchsize"]
        
        for epoch in range(epochs):
            states = []
            state, _ = env.reset()
            stacked_states = deque([state], maxlen=self.hyperparameters["stacked_states_size"])
            for t in count():
                if t < 50:
                    state, _, _, _, _ = env.step(0)
                    stacked_states.append(state)
                    continue

                if t%2 != 0:
                    state, _, _, _, _ = env.step(action.item())
                    stacked_states.append(state)
                    continue
                
                state = self.preprocess(stacked_states)
                state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                states.append(state)

                action = env.action_space.sample()
                state, _, terminated, truncated, _ = env.step(action)
                stacked_states.append(state)

                if terminated or truncated:
                    break
            
            states = torch.cat(states)
            perm = torch.randperm(len(states))
            shuffled = states[perm]
            for i in range(0, len(shuffled) // batch_size * batch_size, batch_size):
                x = shuffled[i:i+batch_size]

                x_hat = self(x)
                loss = F.mse_loss(x_hat, x)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            if epoch%20 == 0:
                title = f"Epoch {epoch}: loss = {loss:.4f}"
                x = x.detach().cpu().numpy()
                x = (x*255.0).astype(np.uint8)

                x_hat = x_hat.detach().cpu().numpy()
                x_hat = (x_hat*255.0).astype(np.uint8)

                _, axes = plt.subplots(2, 2)
                axes[0, 0].imshow(x[0][: , :, 0])
                axes[0, 1].imshow(x[0][: , :, 1])
                axes[1, 0].imshow(x_hat[0][: , :, 0])
                axes[1, 1].imshow(x_hat[0][: , :, 1])
                plt.suptitle(title)
                plt.tight_layout()
                plt.show()
            

In [ ]:
def freeze_conv_layers(module):
    for m in module.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            for p in m.parameters():
                p.requires_grad = False

class CarRacingPPO(PPO):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        autoencoder_path = "autoencoder.pt"
        if not os.path.exists(autoencoder_path):
            # It is quite useful to pre-train the AutoEncoder so it doesn't start from nowhere with the PPO agents
            self.autoencoder.pre_train(self.env, self.device)
            torch.save(self.autoencoder.state_dict(), autoencoder_path)
        else:
            print("Loading ae weights")
            self.autoencoder.load_state_dict(torch.load(autoencoder_path, weights_only=True))

        # Then we fix the convolutional layers so they don't get overwritten, but we keep the Linear layers trainable
        freeze_conv_layers(self.autoencoder)

        # We use the trained convolutional parts for the CriticCNN as well
        self.critic.cnn.load_state_dict(self.autoencoder.encoder.cnn.state_dict())

        # Freeze the critic CNN convolutional layers
        freeze_conv_layers(self.critic.cnn)

        self.optimizer = optim.Adam(
            list(filter(lambda p: p.requires_grad, self.autoencoder.parameters())) +
            list(self.actor.parameters()) +
            list(filter(lambda p: p.requires_grad, self.critic.parameters())),
            lr=self.config["lr"]
        )

    def run(self):
        training_start = datetime.datetime.now()
        rewards = []
        mean_rewards = []
        mean_reward = 0
        total_timesteps = 0
        start = datetime.datetime.now()
        while mean_reward < self.config["mean_reward_stop"]:
            state, _ = self.env.reset()
            stacked_states = deque([state], maxlen=self.config["stacked_states_size"])
            cumulative_reward = 0
            pending_reward = 0
            for t in count():
                if t < 50:
                    state, _, terminated, truncated, _ = self.env.step(0)
                    stacked_states.append(state)
                    if terminated or truncated:
                        break
                    continue

                if t % 2 != 0:
                    state, skip_reward, terminated, truncated, _ = self.env.step(action.item())
                    stacked_states.append(state)
                    pending_reward += skip_reward
                    if terminated or truncated:
                        break
                    continue
                    
                total_timesteps += 1
                state = self.autoencoder.preprocess(stacked_states)
                state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
                with torch.no_grad():
                    encoded_state = F.tanh(self.autoencoder.encoder(state))
                    action, action_log_probability, _, state_value = self.apply_policy(state, encoded_state)

                next_state, reward, terminated, truncated, _ = self.env.step(action.item())
                reward += pending_reward
                pending_reward = 0
                done = terminated or truncated

                self.replay.add(state.detach(), action.detach(), action_log_probability.detach(), state_value.detach(), reward, done)

                cumulative_reward += reward

                state = next_state
                stacked_states.append(state)

                if total_timesteps % self.steps_update_frequency == 0:
                    self.optimize_models()
                    print()
                    print(f"Step {total_timesteps}: mean reward = {mean_reward:.2f} ({datetime.datetime.now()-start})")
                    start = datetime.datetime.now()

                if done:
                    break

            rewards.append(cumulative_reward)
            mean_reward = np.mean(rewards[-self.config["mean_reward_lookback"]:])
            mean_rewards.append(mean_reward)
        
        print(f"\nTotal training time: {datetime.datetime.now()-training_start}")
        return mean_rewards
    
    def evaluate(self):
        fig, ax = plt.subplots()
        im = None

        state, _ = self.env.reset()
        stacked_states = deque([state], maxlen=self.config["stacked_states_size"])
        for t in count():
            if t < 50:
                state, _, _, _, _ = self.env.step(0)
                stacked_states.append(state)
                continue

            if t%2 != 0:
                state, _, _, _, _ = self.env.step(action.item())
                stacked_states.append(state)
                continue

            state = self.autoencoder.preprocess(stacked_states)
            state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0).to(self.device)
            with torch.no_grad():
                encoded_state = F.tanh(self.autoencoder.encoder(state))
                actions_probs = F.softmax(self.actor(encoded_state), dim=-1)
                actions_distribution = torch.distributions.Categorical(actions_probs)
                action = actions_distribution.sample()

            next_state, _, terminated, truncated, _ = self.env.step(action.item())
            done = terminated or truncated
            state = next_state
            stacked_states.append(state)

            frame = self.env.render()
            if im is None:
                im = ax.imshow(frame)
                ax.axis("off")
            else:
                im.set_data(frame)
            clear_output(wait=True)
            display(fig)

            if done:
                break
            
        clear_output(wait=True)
        plt.close(fig)

## 4 - Results
___

### Lunar Lander

In [ ]:
ll_config = {
    "ae_hidden_dim": 64, "ae_output_dim": 3,
    "critic_intermediate_dim": 128,
    "minibatch_size": 64, "lr": 3e-4,
    "K": 4, "episode_update_frequency": 1, "mean_reward_lookback": 50, "mean_reward_stop": 220,
    "gamma": 0.99, "lambda": 0.98, "epsilon": 0.2, "entropy_coeff": 0.01,
    # "ae_hidden_dim": 64, "ae_output_dim": 3, 
    # "critic_intermediate_dim": 64,
    # "K": 25, "episode_update_frequency": 2, "mean_reward_lookback": 50, "mean_reward_stop": 100,
    # "gamma": 0.99, "lambda": 0.95, "epsilon": 0.2, "entropy_coeff": 0.01
    }

In [ ]:
# 1. Classical PPO
ll_config["actor_intermediate_dim"] = 3
ll_classical_ppo = PPO(AutoEncoderClass=AutoEncoder, ActorClass=ActorNN, CriticClass=CriticNN, env=lunarlander_env, config=ll_config)
print(f"Classical actor parameters: {count_parameters(ll_classical_ppo.actor)}")
ll_classical_mean_rewards = ll_classical_ppo.run()

In [ ]:
ll_classical_ppo.evaluate()

In [ ]:
# 2. Qubit PPO
ll_config["actor_intermediate_dim"] = 4
ll_qubit_ppo = PPO(AutoEncoderClass=AutoEncoder, ActorClass=ActorQubitNN, CriticClass=CriticNN, env=lunarlander_env, config=ll_config)
print(f"Qubit actor parameters: {count_parameters(ll_qubit_ppo.actor)}")
ll_qubit_mean_rewards = ll_qubit_ppo.run()

In [ ]:
ll_qubit_ppo.evaluate()

In [ ]:
plt.figure()
plt.axhline(y=ll_config["mean_reward_stop"], color="black")
plt.plot(ll_classical_mean_rewards, label="Classical PPO")
plt.plot(ll_qubit_mean_rewards, label="Qubit-based PPO")
plt.xlabel("Episodes")
plt.ylabel("Rewards Mean Average")
plt.title("Performance")
plt.legend()
plt.show()

### Car Racing

In [ ]:
cr_config = {
    "image_size": 64, "stacked_states_size": 2,
    "ae_hidden_dims": [16, 32, 64, 256], "ae_output_dim": 8, 
    "ae_pretrain_epochs": 300, "ae_pretrain_batchsize": 64,
    "critic_intermediate_dim": [16, 32, 64, 256],
    "minibatch_size": 128, "lr": 1e-4,
    "K": 8, "episode_update_frequency": 4, "mean_reward_lookback": 30, "mean_reward_stop": 850,
    "gamma": 0.99, "lambda": 0.95, "epsilon": 0.2, "entropy_coeff": 0.01
    # "image_size" : 64, "stacked_states_size":3, 
    # "ae_hidden_dims": [8, 16, 32, 128], "ae_output_dim": 8, "ae_pretrain_epochs": 200, "ae_pretrain_batchsize": 32,
    # "critic_intermediate_dim": 64,
    # "K": 25, "episode_update_frequency": 2, "mean_reward_lookback": 50, "mean_reward_stop": 800, 
    # "gamma": 0.99, "lambda": 0.95, "epsilon": 0.2, "entropy_coeff": 0.01
    }
custom_obs_shape = (cr_config["image_size"], cr_config["image_size"], cr_config["stacked_states_size"])

In [ ]:
# 1. Classical PPO
cr_config["actor_intermediate_dim"] = 6
cr_classical_ppo = CarRacingPPO(AutoEncoderClass=ConvolutionalAE, ActorClass=ActorNN, CriticClass=CriticCNN, env=carracing_env, config=cr_config, custom_obs_shape=custom_obs_shape)
print(f"Classical actor parameters: {count_parameters(cr_classical_ppo.actor)}")
cr_classical_mean_rewards = cr_classical_ppo.run()

In [ ]:
cr_classical_ppo.evaluate()

In [ ]:
# 2. Qubit PPO
cr_config["actor_intermediate_dim"] = 5
cr_qubit_ppo = CarRacingPPO(AutoEncoderClass=ConvolutionalAE, ActorClass=ActorQubitNN, CriticClass=CriticCNN, env=carracing_env, config=cr_config, custom_obs_shape=custom_obs_shape)
print(f"Qubit actor parameters: {count_parameters(cr_qubit_ppo.actor)}")
cr_qubit_mean_rewards = cr_qubit_ppo.run()

In [ ]:
cr_qubit_ppo.evaluate()

In [ ]:
plt.figure()
plt.axhline(y=cr_config["mean_reward_stop"], color="black")
plt.plot(cr_classical_mean_rewards, label="Classical PPO")
plt.plot(cr_qubit_mean_rewards, label="Qubit-based PPO")
plt.xlabel("Episodes")
plt.ylabel("Rewards Mean Average")
plt.title("Performance")
plt.legend()
plt.show()

## 5 - Conclusions
___

Bigger environments -> Time constraint but how do performances, memory and params follow? For both classical and qubits?

## 6 - Follow-up
___

Again, Try using other values for actor_intermediate_dim, max stop and lookback

In the next notebook, the three environments seen in the series (Cart Pole, Lunar Lander and Car Racing) will be tested with a Photonic PPO.